# Lab 17: Multi-Memory Agent
This notebook demonstrates a memory-augmented agent with 4 memory types:
- **Short-term memory**: sliding window buffer
- **Long-term profile**: persistent user facts
- **Episodic memory**: task episode log with keyword search
- **Semantic memory**: knowledge base with keyword search

In [ ]:
import sys
sys.path.insert(0, '.')

from memory.short_term import ShortTermMemory
from memory.long_term import LongTermProfile
from memory.episodic import EpisodicMemory
from memory.semantic import SemanticMemory
from agent.state import make_initial_state
from agent.nodes import retrieve_memory, call_llm, save_memory
from agent.graph import SkeletonGraph
from agent.prompt_builder import build_prompt

## 1. Short-Term Memory Demo

In [ ]:
stm = ShortTermMemory(max_turns=6)
for i in range(20):
    stm.add('user', f'turn {i}')
print(f'Buffer size: {len(stm.get_recent())}')
print(f'Expected: 12 (6 turns x 2 entries)')
print(f'Last 4 entries: {stm.get_recent()[-4:]}')

## 2. Long-Term Profile + Conflict Handling

In [ ]:
import tempfile, os
tmpdir = tempfile.mkdtemp()
profile = LongTermProfile(path=os.path.join(tmpdir, 'profile.json'))

# Set initial allergy
profile.set('allergy', 'cow milk')
print(f'After cow milk: {profile.as_dict()}')

# Conflict resolution: overwrite with new value
profile.set('allergy', 'soy milk')
print(f'After soy milk: {profile.as_dict()}')
print(f'Conflict handled: allergy is now "soy milk"')

## 3. Semantic Memory Retrieval

In [ ]:
sem = SemanticMemory(path='data/knowledge_base.json')
results = sem.search('Python venv', top_k=3)
print(f'Found {len(results)} chunks about Python venv:')
for r in results:
    print(f'  - {r[:80]}...')

## 4. Episodic Memory

In [ ]:
epi = EpisodicMemory(path=os.path.join(tmpdir, 'episodes.json'))
epi.add('Fixed docker networking bug', 'Recreated network with docker network create')
hits = epi.search('docker', top_k=3)
print(f'Episodic hits for docker: {hits}')

## 5. Full Agent Pipeline (no API call)

In [ ]:
stm = ShortTermMemory(max_turns=6)
ltm = LongTermProfile(path=os.path.join(tmpdir, 'profile2.json'))
epi = EpisodicMemory(path=os.path.join(tmpdir, 'episodes2.json'))
sem = SemanticMemory(path='data/knowledge_base.json')

# Simulate a conversation
turns = [
    {'role': 'user', 'content': 'My name is Alice'},
    {'role': 'user', 'content': 'How do I set up a Python venv?'},
]

state = make_initial_state()
for turn in turns:
    state['messages'].append(turn)
    state = retrieve_memory(state, stm=stm, ltm=ltm, episodic=epi, semantic=sem)

print('User profile:', state['user_profile'])
print('Semantic hits:', [h[:60] for h in state['semantic_hits']])
print()
prompt = build_prompt(state)
print('Generated prompt (first 500 chars):')
print(prompt[:500])

## 6. Run Benchmark

In [ ]:
from benchmark_runner import run_benchmark
print(run_benchmark())